## 1. 核心思想

这是一种 **"以文档为中心"** 的策略。利用 LLM 预先为每个文档块生成可能的"用户提问"。在检索时，将用户的 Query 与这些"假设问题"进行匹配，而非直接匹配文档内容。

## 2. 核心逻辑伪代码

In [ ]:
def process_document_with_hypothetical_questions(doc_chunk):
    """
    核心逻辑：
    1. 输入：原始文档块
    2. 处理：LLM 生成 N 个假设问题
    3. 输出：N 个 (问题向量, 原始文档) 的键值对
    """

    # Step 1: LLM 逆向生成问题
    hypothetical_questions = llm.generate(
        prompt=f"基于以下文本，生成3个用户可能会问的问题：\n{doc_chunk}"
    )

    results = []

    # Step 2: 遍历生成的问题
    for question in hypothetical_questions:
        # Step 3: 关键点 —— 对“问题”进行向量化，而不是对“文档”
        q_vector = embedding_model.encode(question)

        # Step 4: 存储映射 —— 向量是问题的，但 Payload 是原始文档
        results.append({
            "vector": q_vector,           # 用于检索匹配
            "payload": doc_chunk,         # 检索后返回给 LLM 的上下文
            "metadata": {"source_q": question} # 可选：记录触发的问题
        })

    return results

## 3. 完整实现（Milvus + OpenAI）

构建一个完整的流程：

建库：在 Milvus 中创建集合。

索引：模拟文档 → 生成问题 → 存入 Milvus（存问题向量，挂载原文）。

检索：用户提问 → 匹配最相似的假设问题 → 获取原文 → 生成答案。

### 3.1 配置

In [ ]:
import os
from pymilvus import MilvusClient, DataType
from openai import OpenAI
import numpy as np

# --- 配置部分 ---
os.environ["OPENAI_API_KEY"] = "sk-OdqRypFlfJLKYvLmV6GsG9j0u6CRFBYKErn4xV1Wm0R3q0y9"  # 替换你的 Key
os.environ["OPENAI_BASE_URL"] = "https://api.openai-proxy.org/v1"
client = OpenAI()
milvus_client = MilvusClient(uri="http://111.228.53.183:19530")

COLLECTION_NAME = "hypothetical_rag"
DIMENSION = 1536  # OpenAI text-embedding-3-small 的维度

### 3.2 核心函数封装

In [ ]:
def get_embedding(text):
    """获取文本向量"""
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def generate_hypothetical_questions(doc_text, n=3):
    """
    核心策略：让 LLM 为文档生成假设性问题
    """
    prompt = f"""
    你是以文档为中心的提问者。
    请基于以下文档片段，生成 {n} 个用户可能会询问的问题。
    这些问题应该是简短的、口语化的，并且能通过该文档回答。

    文档内容: "{doc_text}"

    请直接输出问题，每行一个，不要包含序号或其他废话。
    """

    response = client.chat.completions.create(
        model="gpt-4", # 或 gpt-3.5-turbo
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    # 清洗数据，按行分割
    questions = response.choices[0].message.content.strip().split('\n')
    return [q.strip() for q in questions if q.strip()]

### 3.3 准备milvus集合

In [ ]:
if milvus_client.has_collection(COLLECTION_NAME):
    milvus_client.drop_collection(COLLECTION_NAME)

# 创建集合
# vector 存的是 "假设问题" 的向量
# original_text 存的是 "原始文档块"
milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=DIMENSION,
    auto_id=True, # 自动生成主键 ID
    enable_dynamic_field=True # 允许存入 extra 字段
)

print(f"集合 {COLLECTION_NAME} 创建成功。\n")

### 3.4 构建索引

In [ ]:
# --- Step 2: 索引构建阶段 (Document -> Questions -> Vector DB) ---

# 模拟原始文档块
documents = [
    "本智能音箱配备 5000mAh 锂电池，支持 Type-C 接口充电，充满需 2 小时。",
    "设备重置方法：长按顶部的静音键 5 秒钟，直到看到橙色指示灯闪烁。",
    "保修政策：主机保修 1 年，附件（如电源线）保修 3 个月，人为损坏不在保修范围内。"
]


data_rows = []

print("正在处理文档并生成假设问题...")
for doc in documents:
    # 1. 生成假设问题 (策略核心)
    hypo_questions = generate_hypothetical_questions(doc, n=3)
    print(f"\n[原文]: {doc[:20]}...")
    print(f"[生成问题]: {hypo_questions}")

    # 2. 向量化并准备入库
    for q in hypo_questions:
        q_vector = get_embedding(q)

        # 3. 构建数据行
        # 注意：这里是一对多的关系。多个问题的向量，都指向同一个 original_text
        data_rows.append({
            "vector": q_vector,
            "original_text": doc,           # 检索结果需要返回的内容
            "hypothetical_question": q      # 存下来用于调试/验证
        })

### 3.5 插入milvus

In [ ]:
# 4. 批量插入 Milvus
insert_res = milvus_client.insert(
    collection_name=COLLECTION_NAME,
    data=data_rows
)

In [ ]:
print(f"\n成功插入 {insert_res['insert_count']} 条数据 (问题-文档映射)。\n")

## 3.6. 检索
- 定义检索函数

In [ ]:
def rag_search(user_query):
    print(f"用户查询: {user_query}")

    # 1. 向量化用户查询
    query_vector = get_embedding(user_query)

    # 2. 在 Milvus 中检索
    # 这里的逻辑是：用户的 Query 与库里的 "假设问题" 进行相似度匹配
    search_res = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_vector],
        limit=1, # 只取最匹配的一个
        output_fields=["original_text", "hypothetical_question"] # 返回原文
    )

    if not search_res:
        return "未找到相关信息。"

    match = search_res[0][0]
    matched_question = match["entity"]["hypothetical_question"]
    retrieved_doc = match["entity"]["original_text"]
    score = match["distance"]

    print(f"命中假设问题: '{matched_question}' (相似度: {score:.4f})")
    print(f"召回原始文档: {retrieved_doc}")

    # 3. 生成最终答案 (Standard Generation)
    final_prompt = f"""
    基于以下参考信息回答用户问题。

    参考信息: {retrieved_doc}
    用户问题: {user_query}
    """

    answer_response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": final_prompt}]
    )

    return answer_response.choices[0].message.content

## 3.7 检索
- 测试

In [9]:
# 测试案例 1：语义模糊的查询
# 用户问的是“没电了多久能好”，这与文档中的“充满需 2 小时”语义差距较大，
# 但LLM生成的假设问题可能包含“多久能充满”，从而匹配成功。
answer1 = rag_search("没电了多久能好？")
print(f"\n>> RAG 回答: {answer1}\n")


# # 测试案例 2：特定操作
# answer2 = rag_search("怎么恢复出厂设置？") # 原文写的是“重置”，通过“恢复出厂”这个假设问题关联
# print(f"\n>> RAG 回答: {answer2}\n")

用户查询: 没电了多久能好？
命中假设问题: '这个智能音箱充满电需要多长时间？' (相似度: 0.4718)
召回原始文档: 本智能音箱配备 5000mAh 锂电池，支持 Type-C 接口充电，充满需 2 小时。

>> RAG 回答: 智能音箱充满电需要2小时。

